# Fink/LSST — Alert Stream Statistics (`api/v1/statistics`)

This notebook explores the **nightly and cumulative statistics** provided by the Fink/LSST broker via the `api/v1/statistics` endpoint.

The endpoint returns per-night counters for the full alert stream processed by Fink, including:
- total number of alerts and unique objects per night
- per-tag / per-classifier breakdown
- observing conditions metadata (if available)

**API reference** (from `swagger.json`):
```
GET/POST  https://api.lsst.fink-portal.org/api/v1/statistics

Parameters:
  date         : YYYYMMDD | YYYYMM | YYYY | "" (empty = all nights)
  columns      : comma-separated, e.g. 'f:alerts,f:night'  (default: all)
  output-format: json | csv | parquet | votable  (default: json)
```

---
**Dependencies**: `requests`, `pandas`, `numpy`, `matplotlib`

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from io import StringIO
import os
import warnings
warnings.filterwarnings('ignore')

print(f'pandas  version : {pd.__version__}')
print(f'numpy   version : {np.__version__}')

In [ ]:
# ---- API endpoint ----
FINK_API   = 'https://api.lsst.fink-portal.org'
STATS_URL  = f'{FINK_API}/api/v1/statistics'

# ---- Plot style ----
plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# where are stored the figures
pathfigs = "figs_FINKAPISTAT04"
prefix = "stat04"
if not os.path.exists(pathfigs):
    os.makedirs(pathfigs) 
figtype = ".png"

# where are stored the data
pathdata = "data_FINKAPISTAT04"
if not os.path.exists(pathdata):
    os.makedirs(pathdata) 

## 2. Helper functions

In [ ]:
def fetch_statistics(date: str = '', columns: str = '', output_format: str = 'csv') -> pd.DataFrame:
    """
    Query the Fink /api/v1/statistics endpoint.

    Parameters
    ----------
    date : str
        Observing date filter:
        - ''       -> all available nights
        - 'YYYY'   -> one year
        - 'YYYYMM' -> one month
        - 'YYYYMMDD' -> one night
    columns : str
        Comma-separated column names (e.g. 'f:alerts,f:night').
        Empty string transfers all columns.
    output_format : str
        'csv' (default), 'json', 'parquet', 'votable'.

    Returns
    -------
    pd.DataFrame  sorted by night (ascending).
    """
    params = {
        'date'         : date,
        'output-format': output_format,
    }
    if columns:
        params['columns'] = columns

    r = requests.get(STATS_URL, params=params, timeout=60)
    r.raise_for_status()

    if output_format == 'csv':
        df = pd.read_csv(StringIO(r.text))
    elif output_format == 'json':
        df = pd.DataFrame(r.json())
    else:
        raise ValueError(f'Unsupported format: {output_format}')

    # Rename columns: strip the 'f:' Fink prefix for convenience
    df.columns = [c.replace('f:', '') for c in df.columns]

    # Parse the night column to datetime (format: YYYYMMDD integer or string)
    if 'night' in df.columns:
        df['night'] = pd.to_datetime(df['night'].astype(str), format='%Y%m%d')
        df = df.sort_values('night').reset_index(drop=True)

    return df


def pretty_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Return a summary DataFrame with column name, dtype, and first non-null value."""
    rows = []
    for col in df.columns:
        first_val = df[col].dropna().iloc[0] if df[col].dropna().shape[0] > 0 else None
        rows.append({'column': col, 'dtype': str(df[col].dtype), 'example': first_val})
    return pd.DataFrame(rows)

print('Helper functions defined.')

## 3. Fetch all available statistics

In [ ]:
# Retrieve ALL nights (date='' means no filter)
df = fetch_statistics(date='')

print(f'Shape : {df.shape}  ({df.shape[0]} nights, {df.shape[1]} columns)')
if 'night' in df.columns:
    print(f'Date range : {df["night"].min().date()}  ->  {df["night"].max().date()}')
df.head(3)

### 3.1  Schema inspection

In [ ]:
schema = pretty_schema(df)
print(f'Available columns ({len(schema)}):')
pd.set_option('display.max_rows', 80)
schema

In [ ]:
# Quick summary statistics
numeric_cols = df.select_dtypes(include='number').columns.tolist()
df[numeric_cols].describe().T.sort_index()

## 4. Nightly alert counts

In [ ]:
# Identify the main alert-count column (API may call it 'alerts' or 'n_alerts')
alert_col  = next((c for c in df.columns if 'alert' in c.lower() and 'unique' not in c.lower()), None)
object_col = next((c for c in df.columns if 'unique' in c.lower() or 'object' in c.lower()), None)

print(f'Alert column  : {alert_col}')
print(f'Object column : {object_col}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# -- panel 1: nightly counts --
ax = axes[0]
if alert_col:
    ax.bar(df['night'], df[alert_col], width=0.8, color='steelblue', alpha=0.8, label='Alerts / night')
    ax.set_ylabel('Alerts per night')
    ax.set_title('Fink/LSST — Nightly alert counts')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(fontsize=9)

# -- panel 2: unique objects --
ax2 = axes[1]
if object_col:
    ax2.bar(df['night'], df[object_col], width=0.8, color='darkorange', alpha=0.8, label='Unique objects / night')
    ax2.set_ylabel('Unique objects per night')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax2.legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())

fig.autofmt_xdate(rotation=35, ha='right')
plt.tight_layout()
figfilename = f"{pathfigs}/{prefix}_pernightalertcount{figtype}"
plt.savefig(figfilename)
plt.show()

## 5. Cumulative totals over time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

if alert_col:
    cum_alerts = df[alert_col].cumsum()
    ax.fill_between(df['night'], cum_alerts, alpha=0.25, color='steelblue')
    ax.plot(df['night'], cum_alerts, color='steelblue', lw=1.8, label='Cumulative alerts')

if object_col:
    ax2 = ax.twinx()
    cum_obj = df[object_col].cumsum()
    ax2.plot(df['night'], cum_obj, color='darkorange', lw=1.8, ls='--', label='Cumulative objects')
    ax2.set_ylabel('Cumulative unique objects', color='darkorange')
    ax2.tick_params(axis='y', labelcolor='darkorange')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.2f} M'))
    ax2.legend(loc='upper left', fontsize=9)

ax.set_title('Fink/LSST — Cumulative alert stream')
ax.set_ylabel('Cumulative alerts', color='steelblue')
ax.tick_params(axis='y', labelcolor='steelblue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.2f} M'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.legend(loc='upper right', fontsize=9)

fig.autofmt_xdate(rotation=35, ha='right')
plt.tight_layout()
figfilename = f"{pathfigs}/{prefix}_cumulnightalertcount{figtype}"
plt.savefig(figfilename)
plt.show()

if alert_col:
    print(f'Total alerts processed : {int(df[alert_col].sum()):,}')
if object_col:
    print(f'Total unique objects   : {int(df[object_col].sum()):,}')

## 6. Per-tag / per-classifier breakdown

In [ ]:
# Identify classifier / tag columns  (those that are neither 'night' nor the main counters)
exclude = {'night', alert_col, object_col}
tag_cols = [c for c in numeric_cols if c not in exclude]
print(f'Tag / classifier columns ({len(tag_cols)}):')
for c in tag_cols:
    tot = df[c].sum()
    print(f'  {c:<45s}  total = {int(tot):>12,}')

In [ ]:
if tag_cols:
    # -- Pie chart: total distribution across tags --
    totals = df[tag_cols].sum().sort_values(ascending=False)
    # Keep top-N and lump the rest into 'other'
    TOP_N = 10
    top   = totals.iloc[:TOP_N]
    other = totals.iloc[TOP_N:].sum()
    if other > 0:
        top['other'] = other

    fig, ax = plt.subplots(figsize=(9, 9))
    wedges, texts, autotexts = ax.pie(
        top.values,
        labels=top.index,
        autopct=lambda p: f'{p:.1f}%' if p > 1.5 else '',
        startangle=90,
        counterclock=False,
        pctdistance=0.80,
        textprops={'fontsize': 8},
    )
    for at in autotexts:
        at.set_fontsize(7.5)
    ax.set_title(f'Fink/LSST — Tag distribution (top {TOP_N})', fontsize=11, pad=15)
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_disktalertfractions{figtype}"
    plt.savefig(figfilename)
    plt.show()

In [ ]:
if tag_cols:
    # -- Stacked bar: nightly tag evolution (top 8 tags) --
    top8 = totals.iloc[:8].index.tolist()

    fig, ax = plt.subplots(figsize=(14, 5))
    bottom = np.zeros(len(df))
    cmap   = plt.get_cmap('tab10')

    for i, col in enumerate(top8):
        vals = df[col].fillna(0).values
        ax.bar(df['night'], vals, bottom=bottom, width=0.8,
               label=col, color=cmap(i), alpha=0.85)
        bottom += vals

    ax.set_title('Fink/LSST — Nightly tag breakdown (top 8 tags)')
    ax.set_ylabel('Alerts per night')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.legend(fontsize=7, loc='upper left', ncol=2, framealpha=0.9)

    fig.autofmt_xdate(rotation=35, ha='right')
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_tagstackednightalertcount{figtype}"
    plt.savefig(figfilename)
    plt.show()

## 7. Monthly aggregation

In [ ]:
df_monthly = df.set_index('night').resample('ME').sum(numeric_only=True).reset_index()
df_monthly['month_label'] = df_monthly['night'].dt.strftime('%Y-%m')

print(f'Monthly rows: {len(df_monthly)}')
if alert_col:
    print(df_monthly[['month_label', alert_col]].to_string(index=False))

In [ ]:
if len(df_monthly) > 0 and alert_col:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(df_monthly))
    bars = ax.bar(x, df_monthly[alert_col], color='steelblue', alpha=0.85, width=0.6)

    # Value labels on bars
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h * 1.01,
                    f'{h/1e3:.1f}k', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(df_monthly['month_label'], rotation=45, ha='right', fontsize=8)
    ax.set_title('Fink/LSST — Monthly total alerts')
    ax.set_ylabel('Alerts per month')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e3:.0f}k'))
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_permonthalertcount{figtype}"
    plt.savefig(figfilename)
    plt.show()

## 8. Per-tag cumulative growth

In [ ]:
if tag_cols:
    top_tags = totals.iloc[:6].index.tolist()

    fig, ax = plt.subplots(figsize=(14, 5))
    cmap = plt.get_cmap('tab10')

    for i, col in enumerate(top_tags):
        cum = df[col].fillna(0).cumsum()
        ax.plot(df['night'], cum, lw=1.8, color=cmap(i), label=col)

    ax.set_title('Fink/LSST — Cumulative alerts per tag (top 6)')
    ax.set_ylabel('Cumulative count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.legend(fontsize=8, loc='upper left', framealpha=0.9)

    fig.autofmt_xdate(rotation=35, ha='right')
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_cumulpertagalertcount{figtype}"
    plt.savefig(figfilename)
    plt.show()
    

## 9. Rolling 7-day average — alert rate

In [ ]:
if alert_col and len(df) > 7:
    roll7 = df[alert_col].rolling(7, center=True, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(df['night'], df[alert_col], width=0.8, color='steelblue', alpha=0.4, label='Nightly')
    ax.plot(df['night'], roll7, color='navy', lw=2.0, label='7-night rolling mean')

    ax.set_title('Fink/LSST — Nightly alerts with 7-night rolling mean')
    ax.set_ylabel('Alerts per night')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.legend(fontsize=9)

    fig.autofmt_xdate(rotation=35, ha='right')
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_mean7dayrollingalertcount{figtype}"
    plt.savefig(figfilename)
    plt.show()

## 10. Night-of-week patterns (observability)

In [ ]:
if alert_col:
    df['dow'] = df['night'].dt.day_of_week          # 0=Mon, 6=Sun
    df['week_num'] = df['night'].dt.isocalendar().week.astype(int)

    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    dow_mean = df.groupby('dow')[alert_col].mean()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([dow_labels[i] for i in dow_mean.index], dow_mean.values,
           color='mediumseagreen', alpha=0.85, edgecolor='white')
    ax.set_title('Fink/LSST — Mean nightly alerts per day of week')
    ax.set_ylabel('Mean alerts per night')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_lastweekperdayalertcount{figtype}"
    plt.savefig(figfilename)
    plt.show()

## 11. Heatmap: alerts per week × day-of-week

In [ ]:
if alert_col and len(df) > 14:
    pivot = df.pivot_table(index='week_num', columns='dow', values=alert_col, aggfunc='sum')
    pivot.columns = [dow_labels[c] for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.28)))
    im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd',
                   interpolation='nearest')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'W{w}' for w in pivot.index], fontsize=7)
    ax.set_title('Fink/LSST — Alert heatmap (week × day-of-week)')

    plt.colorbar(im, ax=ax, label='Alerts', shrink=0.6)
    plt.tight_layout()
    figfilename = f"{pathfigs}/{prefix}_heatmapalertcount{figtype}"
    plt.savefig(figfilename)
    plt.show()

## 12. Single-night deep-dive

In [ ]:
# Fetch the statistics for the most productive night and display full row
if alert_col:
    best_idx  = df[alert_col].idxmax()
    best_night = df.loc[best_idx, 'night']
    best_date  = best_night.strftime('%Y%m%d')

    print(f'Most productive night : {best_night.date()}  ({int(df.loc[best_idx, alert_col]):,} alerts)')

    df_night = fetch_statistics(date=best_date)
    df_night.T.rename(columns={0: best_date})

## 13. Summary table

In [ ]:
summary_rows = []

if alert_col:
    summary_rows += [
        ('Total alerts',          f"{int(df[alert_col].sum()):,}"),
        ('Mean alerts / night',   f"{df[alert_col].mean():.0f}"),
        ('Median alerts / night', f"{df[alert_col].median():.0f}"),
        ('Max alerts (one night)', f"{int(df[alert_col].max()):,}  ({df.loc[df[alert_col].idxmax(),'night'].date()})"),
        ('Nights with > 0 alerts', f"{(df[alert_col] > 0).sum()}"),
    ]

if object_col:
    summary_rows.append(('Total unique objects', f"{int(df[object_col].sum()):,}"))

if 'night' in df.columns:
    summary_rows += [
        ('First night', str(df['night'].min().date())),
        ('Last night',  str(df['night'].max().date())),
        ('Total nights in DB', str(len(df))),
    ]

summary_df = pd.DataFrame(summary_rows, columns=['Metric', 'Value'])
summary_df